<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/what_are_llms_and_generative_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 What are LLMs & Generative AI?

Generative AI **creates** new content; an LLM is the text-specific branch of it. This notebook proves both ideas with code running against a model on **your own machine** (via [LM Studio](https://lmstudio.ai/)) — no API key, no internet.

> ⚠️ **Run this locally**, not in Colab — Colab's cloud VM can't reach `localhost` on your laptop.

**You'll see in code:** next-token prediction happening live, one word at a time, and the same function call handling Q&A, summarization, code, and reasoning.

## Generative AI, in one line

It **creates** new content (text, images, code, audio) instead of just analyzing or classifying existing data — that's what separates a chatbot from a spam filter.

## LLMs, in one line

An **LLM** is Generative AI trained specifically on language: it learns to predict "what word comes next" from huge amounts of text, and that one skill, at scale, is enough to answer, write, summarize, and reason. Demo 3 below proves it with code.

## LLM vs. Generative AI

**LLM is a subset of Generative AI, not a synonym.** Every LLM is Generative AI (it generates text). Not every Generative AI is an LLM — Midjourney generates images, but it wasn't built on language.

## The landscape (a snapshot — it shifts every few months)

**LLMs (text):** OpenAI (GPT), Anthropic (Claude), Google DeepMind (Gemini), Meta (LLaMA, open-weight), Mistral, xAI (Grok), DeepSeek, Alibaba (Qwen).
**Other Generative AI:** Midjourney / Stable Diffusion (images), Sora / Veo (video), Suno (audio), GitHub Copilot (code).

The model we run locally below sits in the first list — it's an LLM: text in, text out.

## ✅ Prerequisites

- [LM Studio](https://lmstudio.ai/) installed on this machine
- At least one chat model downloaded inside LM Studio (pick a small one, e.g. a 1B–4B parameter model, so it runs fast even on a laptop)
- Python 3.9+ with your bootcamp `venv` activated

## 🚀 Step 1 — Start the Local Server in LM Studio

Open **LM Studio** → **Developer** tab (`</>` icon) → load a downloaded chat model → **Start Server**. It exposes an **OpenAI-compatible API** (default `http://localhost:1234/v1`), so we reuse the `openai` SDK, just pointed at your machine instead of OpenAI's. No real API key is checked — any non-empty string works.

## ⚙️ Step 2 — Install the SDK

In [ ]:
%pip install -q openai

## 🔌 Step 3 — Connect to Your Local Server

Change `BASE_URL` below if LM Studio's Developer tab shows a different port.

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"

client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

print(f"✅ Client configured for {BASE_URL}")

## 🔍 Step 4 — Discover Which Model Is Loaded

A local model's ID depends on what you downloaded, so we ask the server instead of hardcoding a name.

In [ ]:
models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]

print("Models loaded in LM Studio:")
for model_id in chat_models:
    print(" -", model_id)

if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"\n✅ Using MODEL: {MODEL}")

## 🎨 Demo 1 — Generative AI in Action

Ask for two things that never existed before this call — that's the defining trait of "generative."

In [ ]:
# Same call shape, two unrelated creative prompts — nothing here is looked up, it's generated
for prompt in [
    "Write a two-line poem about a laptop that runs an AI model offline.",
    "Invent a short, catchy name for a bootcamp that teaches AI in 5 days, with one sentence explaining it.",
]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
    )
    print(f"Prompt: {prompt}\n→ {response.choices[0].message.content.strip()}\n")

Neither the poem nor the name existed before this call — a classifier could only ever hand back one of a fixed set of labels; this model **created** both.

## 🔮 Demo 2 — Watching Next-Token Prediction

An LLM doesn't compose a whole reply at once — it repeatedly asks "given everything so far, what's the single most likely next token?", appends it, and repeats. Below, we force the model to output exactly **one token per call** and feed its own output back in — a slow-motion replay of what normally happens instantly inside one request.

In [ ]:
text = "The best way to learn AI is to"
print(text, end="", flush=True)

for _ in range(15):
    response = client.chat.completions.create(
        model=MODEL,
        # system prompt + max_tokens=3 forces "one word only" so we can watch generation step by step
        messages=[
            {"role": "system", "content": "Continue the user's sentence. Output ONLY the next word, nothing else."},
            {"role": "user", "content": text},
        ],
        max_tokens=3,
        temperature=0.0,
    )
    next_piece = response.choices[0].message.content.strip().split()
    if not next_piece:
        break
    next_word = next_piece[0]
    text += " " + next_word  # feed the model's own output back in as the growing prompt
    print(" " + next_word, end="", flush=True)

print("\n\nFinal sentence:", text)

That loop **is** next-token prediction, made visible. Normally an LLM does this hundreds of times per response internally, in a fraction of a second — which is why streaming shows text appearing token-by-token instead of all at once.

## 🧩 Demo 3 — One Skill, Many Uses

Same `client.chat.completions.create` call shape for all four tasks below — only the prompt changes.

In [ ]:
tasks = {
    "Answering a question": "What is the capital of France?",
    "Summarizing": "Summarize in one sentence: The bootcamp runs for five days, covering LLM fundamentals, "
                    "prompt engineering, RAG, agents, and a final capstone project where students build their own AI app.",
    "Writing code": "Write a one-line Python function that returns the square of a number.",
    "Reasoning": "If a train leaves at 3pm and takes 2 hours 30 minutes, what time does it arrive? Answer in one line.",
}

for task_name, prompt in tasks.items():
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        temperature=0.0,
    )
    print(f"[{task_name}]\n{response.choices[0].message.content.strip()}\n")

Four different-looking tasks, one mechanism underneath: predict the next token, append it, repeat.

## 🎯 Recap

| Concept | What you learned |
|---|---|
| Generative AI | Creates new content instead of analyzing/classifying existing data |
| LLM | The language branch of Generative AI — predicts and generates text |
| The relationship | Every LLM is Generative AI; not every Generative AI is an LLM |
| How it works | Repeatedly predicts the next token, appends it, repeats — you watched this happen live |
| Why it's powerful | That one skill, at scale, drives Q&A, summarizing, coding, and reasoning — proven with one identical call across four tasks |

**Next:** `llm_fundamentals_lmstudio.ipynb` (or the cloud-provider notebooks) — sampling parameters, system instructions, multi-turn chat, and streaming.